In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm
import shutil
import json
import pickle
import time

try:
    import xmltodict
except:
    ! pip install xmltodict

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning

from preprocessing import Preprocessing
from api import ParsePayload

(CVXPY) Dec 16 09:12:49 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Dec 16 09:12:49 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


#### Functions

#### Constants

In [2]:
str_splitter = '/'

# project
str_project = os.getcwd().split(str_splitter)[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split(str_splitter)[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split(str_splitter)[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

# tiers
str_tiers = """
{
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500,
}
"""
str_tiers = str_tiers.replace(' ','')

# pricing
flt_pct_threshold = 0.10
int_dollars_round_fees = 1
flt_avg_life = 2.3
flt_equity_intercept = 0.04
flt_equity_slope = 0.80
flt_securitization = 0.0595
flt_late_fee_income = 0.0042
flt_state_rate_cap = 1.0
flt_cnl_scaler = 0.8039
flt_prop_c = 0.5

Project: 20241112-simple-model-test
Task: 09_15_in_60
Subtask: 06_parser


#### Make output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Read payload

In [4]:
str_filename = 'request_8473752_12.json'
str_local_path = f'./input/{str_filename}'
dict_json_request = json.load(open(str_local_path))['request']
str_json_request = json.dumps(dict_json_request)

#### Copy preprocessing script

In [5]:
str_filename = 'preprocessing.py'
str_origin = f'../../08_prep_data/{str_filename}'
str_destination = f'./{str_filename}'
shutil.copyfile(str_origin, str_destination)

'./preprocessing.py'

#### Get preprocessing model and attributes

In [6]:
# import
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'../../08_prep_data/output/{str_filename}'
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# get attributes
flt_quantile = cls_model_preprocessing.flt_quantile
flt_income_max = cls_model_preprocessing.flt_income_max

# get imputation dictionary
str_filename = 'dict_impute.pkl'
str_local_path = f'../02_model/output/{str_filename}'
dict_impute = pickle.load(open(str_local_path, 'rb'))

# get bins for scorecard
str_filename = 'dict_bins.pkl'
str_local_path = f'../02_model/output/{str_filename}'
dict_bins = pickle.load(open(str_local_path, 'rb'))

#### Initialize class

In [7]:
# init
cls_model_preprocessing = Preprocessing(
    dict_impute=dict_impute,
    dict_bins=dict_bins,
)
# assign
cls_model_preprocessing.flt_income_max = flt_income_max

# save
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_model_preprocessing, open(str_local_path, 'wb'))

#### Get inference model

In [8]:
str_filename = 'cls_model_inference_ml_logistic_scorecard.pkl'
str_local_path = f'../02_model/output/{str_filename}'
cls_model_inference = pickle.load(open(str_local_path, 'rb'))

#### Get quantiles from the model

In [9]:
str_filename = 'arr_score_quantiles_yhat_ml_logistic_scorecard.pkl'
str_local_path = f'../03_get_predictions/output/{str_filename}'
arr_quantiles_model = pickle.load(open(str_local_path, 'rb'))

#### Get quantiles for pricing

In [10]:
str_filename = 'arr_score_quantiles_ecnl.pkl'
str_local_path = f'../04_pricing_distribution/output/{str_filename}'
arr_score_quantiles_pricing = pickle.load(open(str_local_path, 'rb'))

#### Adverse action dictionary

In [11]:
df_aa = pd.read_csv('./input/df_aa.csv')
dict_aa = dict(zip(df_aa['feature'], df_aa['reason']))

# ensure captialized
dict_aa = {key: val.capitalize() for key, val in dict_aa.items()}

# pickle
str_filename = 'dict_aa.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(dict_aa, open(str_local_path, 'wb'))

#### Initialize class

In [12]:
cls_parser = ParsePayload(
    cls_model_preprocessing=cls_model_preprocessing,
    cls_model_inference=cls_model_inference,
    arr_quantiles_model=arr_quantiles_model,
    arr_score_quantiles_pricing=arr_score_quantiles_pricing,
    dict_aa=dict_aa,
    str_tiers=str_tiers,
)

#### Start time

In [13]:
time_start = time.perf_counter()

#### Parse payload

In [14]:
# get data
cls_parser.get_data(str_request=str_json_request)
# engineer pmt hx
cls_parser.engineer_pmt_hx()
# preprocessing
cls_parser.preprocessing()
# get predictions
cls_parser.get_predictions()
# interpolate
cls_parser.interpolate()
# adverse action
cls_parser.adverse_action()
# counter offers
cls_parser.counter_offers()
# generate response
cls_parser.generate_response()

Getting data...
[8473752104567021]: Engineering payment history...
[8473752104567021]: Preprocessing data...
Masking negative values to NaN...


100%|██████████| 31/31 [00:00<00:00, 1820.01it/s]


Replacing zeros...


100%|██████████| 2/2 [00:00<00:00, 1311.54it/s]


Rounding values...


...

Binning values...


100%|██████████| 25/25 [00:00<00:00, 1338.56it/s]


[8473752104567021]: Getting predictions...
[8473752104567021]: Interpolating...
[8473752104567021]: Getting adverse action...


100%|██████████| 25/25 [00:00<00:00, 1820.82it/s]
1it [00:00, 1576.81it/s]
100%|██████████| 1/1 [00:00<00:00, 3688.92it/s]

[8473752104567021]: Getting counter offers...
BK: False
Vehicle Class: Class 3
Dealer Type: Independent
Dealer State: North Carolina
[8473752104567021]: Pricing counter offers...
Applicant Label: nonBK-Independent
Equity Intercept: 0.04
Equity Slope: 0.8
Securitization: 0.0595
Vehicle Class: Class 3
[8473752104567021]: Showing counter offers...
Counter threshold: 0.33599999999999997
Initial ECNL: 0.1954
Approved (T/F): True
Tier: B
Initial offer Approved: 0.1954
Original Values:
{'LTV': 1.3032604373757455, 'APR': 0.2228171557340486, 'Fees': 450.0, 'Tier': 'B'}
Original LTV: 1.3033
Minimum LTV: 1.1729
Original APR: 0.2228
Original Fees: 450.0000
APR + Fees Threshold: 100
There are 0 counters
Best Counter Offer: 0
Original APR: 0.2228171557340486
Original Net Discount: 450.0
[8473752104567021]: Generating response...


#### End time

In [15]:
time_end = time.perf_counter()
flt_sec = time_end - time_start
print(f'Time to parse: {flt_sec:0.4f} sec.')

Time to parse: 0.2594 sec.


#### Show response

In [16]:
cls_parser.dict_response

{'Request_id': '',
 'Zaml_processing_id': '',
 'Response': [{'Model_name': 'prestige-gen-xiii',
   'Model_version': 'v1',
   'Results': [{'Row_id': 8473752104567021,
     'Score_pd': 0.122436521,
     'Score_ecnl_mod': 0.1954414609,
     'APR': 0.2228171557,
     'Net_discount': 450.0,
     'Key_factors': ['Excessive debt to income',
      'Insufficient income',
      'Excessive account balances',
      'Insufficient credit file, length of credit',
      'Negative payment history'],
     'Outlier_score': 0.0,
     'Dict_tiers': "\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"}],
   'Errors': [],
   'CounterOffers': [{'Offer': 0,
     'ECNL': 0.1954414609207049,
     'Decision': 'Approved',
     'APR': 0.2228171557340486,
     'NetDiscount': 450.0,
     'CurrentLTV': 1.3032604373757455,
     'MaxLTV': 1.3032604373757455}]}]}

#### Save

In [17]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))